# ORIE 5135 Project — Exam Scheduling ILP Implementation

This notebook implements and runs the three ILP formulations discussed in the report:

1. **F1:** Natural assignment formulation.
2. **F2:** Clique-strengthened assignment formulation.
3. **F3:** Extended grouping formulation.

The report contains the full mathematical formulations and discussion. This notebook focuses on implementation details, data loading, model construction, LP/MIP solving, and result collection.

The code uses **Python + gurobipy**, reads instances directly from JSON files, and runs:

- **Dataset 1:** F1, F2, and F3 on `instance_01.json` to `instance_10.json`.
- **Dataset 2:** F1 and F2 on `instance_01.json` to `instance_09.json`.

F3 is not run on Dataset 2 because it requires enumerating all feasible exam groupings, which is computationally impractical for larger instances.

## Global imports and configuration

This section imports the required packages and defines the global settings used throughout the notebook.

The expected folder structure is:

```text
project_folder/
    ORIE5135_Project_submit.ipynb
    dataset1/
        instance_01.json
        ...
        instance_10.json
    dataset2/
        instance_01.json
        ...
        instance_09.json

In [1]:
import os
import json
import time
from pathlib import Path

import pandas as pd
import gurobipy as gp
from gurobipy import GRB

# -----------------------------
# Final experiment configuration
# -----------------------------
DATASET1_DIR = Path("dataset1")
DATASET2_DIR = Path("dataset2")

DATASET1_INSTANCES = list(range(1, 11))  # instance_01.json to instance_10.json
DATASET2_INSTANCES = list(range(1, 10))   # instance_01.json to instance_09.json

DATASET1_TIME_LIMIT = 120  # seconds per MIP solve, as required by the project
DATASET2_TIME_LIMIT = 600  # seconds per MIP solve, as stated in the project PDF
LP_TIME_LIMIT = 120        # seconds per LP relaxation solve

# Final submission mode: run all required experiments by default.
RUN_DATASET1 = True
RUN_DATASET2 = True
RESUME_FROM_CHECKPOINTS = True

# Use new checkpoint names so old, invalid, full-clique results are not reused accidentally.
D1_CHECKPOINT = Path("results_dataset1_final_clean_checkpoint.csv")
D2_CHECKPOINT = Path("results_dataset2_final_clean_checkpoint.csv")

# Practical clique strengthening controls. We keep all pairwise conflict constraints and add selected
# clique inequalities. This avoids the full maximal-clique explosion while still implementing F2.
MIN_CLIQUE_SIZE = 3
MAX_CLIQUES_DATASET1 = 1000
MAX_CLIQUES_DATASET2 = 500

# MIP-size reduction: for F1/F2 integer solves only, keep only the number of slots used by a greedy
# feasible schedule. This is a valid upper bound and does not change the integer optimum.
USE_GREEDY_SLOT_UPPER_BOUND_FOR_MIP = True

# Gurobi log control. Set to 1 if detailed solver logs are needed.
OUTPUT_FLAG = 0

## Shared helper functions

This section defines reusable helper functions for the implementation:

- Load one JSON instance from `dataset1/` or `dataset2/`.
- Convert conflict edges from 1-based JSON indexing to 0-based Python indexing.
- Build graph adjacency structures used by F1, F2, and F3.
- Generate selected clique inequalities for F2.
- Construct greedy feasible schedules for warm starts and slot upper bounds.
- Solve LP relaxation and MIP models.
- Collect solver statistics into result rows.

Implementation choices used in later sections:

- Each MIP solve uses the required project time limit.
- LP relaxations are solved separately and reported explicitly.
- For F1/F2 MIP solves, a greedy feasible schedule gives an upper bound `U` on the number of used slots, so the MIP uses only slots `1, ..., U`. This reduces symmetry and model size without changing the integer optimum.
- F2 keeps all pairwise conflict constraints and adds selected clique inequalities. This matches the report's practical clique-strengthened formulation.
- Checkpoint CSV files are written after each formulation-instance run so interrupted experiments can resume.

In [2]:
def instance_path(dataset_dir, k):
    return dataset_dir / f"instance_{k:02d}.json"


def load_instance(path):
    """Load one JSON instance and convert edges from 1-based to 0-based indexing."""
    path = Path(path)
    with open(path, "r") as f:
        data = json.load(f)
    m = int(data["m"])
    n = int(data["n"])
    R = int(data["R"])
    r = [int(v) for v in data["r"]]
    edges = [(int(i)-1, int(j)-1) for i, j in data["edges"]]
    edges = [(min(i, j), max(i, j)) for i, j in edges if i != j]
    edges = sorted(set(edges))
    assert len(r) == m
    assert all(0 <= i < m and 0 <= j < m for i, j in edges)
    assert all(req <= R for req in r)
    return {"m": m, "n": n, "R": R, "r": r, "edges": edges, "path": str(path)}


def build_adjacency(m, edges):
    adj = [set() for _ in range(m)]
    for i, j in edges:
        adj[i].add(j)
        adj[j].add(i)
    return adj


def build_adj_masks(m, edges):
    """Build bit masks for adjacency tests in the F3 grouping enumeration.

    adj_masks[i] has bit k equal to 1 if exam i conflicts with exam k. This lets the
    backtracking enumerator test whether a candidate exam conflicts with the current
    partial group using one fast bitwise operation.
    """
    adj_masks = [0] * m
    for i, j in edges:
        adj_masks[i] |= (1 << j)
        adj_masks[j] |= (1 << i)
    return adj_masks


def status_name(status):
    return {
        GRB.OPTIMAL: "Optimal",
        GRB.TIME_LIMIT: "TimeLimit",
        GRB.INFEASIBLE: "Infeasible",
        GRB.INF_OR_UNBD: "InfOrUnbd",
        GRB.UNBOUNDED: "Unbounded",
        GRB.INTERRUPTED: "Interrupted",
        GRB.NUMERIC: "Numeric",
        GRB.SUBOPTIMAL: "Suboptimal",
    }.get(status, str(status))


def safe_attr(model, attr, default=None):
    try:
        return getattr(model, attr)
    except Exception:
        return default


def configure_model(model, time_limit=None, is_mip=True):
    """Apply solver parameters that control runtime without changing the mathematical formulation."""
    model.Params.OutputFlag = OUTPUT_FLAG
    if time_limit is not None:
        model.Params.TimeLimit = float(time_limit)
    model.Params.Threads = 0
    model.Params.Presolve = 2
    model.Params.Symmetry = 2
    if is_mip:
        model.Params.MIPFocus = 1
        model.Params.Heuristics = 0.30
        model.Params.Cuts = 2
    return model


def extract_solution_metrics(model):
    """Return solver metrics robustly, including time-limit cases."""
    sol_count = int(safe_attr(model, "SolCount", 0) or 0)
    has_incumbent = sol_count > 0
    obj_val = safe_attr(model, "ObjVal", None) if has_incumbent else None
    obj_bound = safe_attr(model, "ObjBound", None)
    gap = safe_attr(model, "MIPGap", None) if has_incumbent else None
    return {
        "status_code": int(model.Status),
        "status": status_name(model.Status),
        "sol_count": sol_count,
        "objective": obj_val,
        "best_bound": obj_bound,
        "mip_gap": gap,
        "node_count": safe_attr(model, "NodeCount", None),
        "solve_time": safe_attr(model, "Runtime", None),
        "optimality": model.Status == GRB.OPTIMAL,
    }


def solve_lp_model(model, time_limit=LP_TIME_LIMIT):
    configure_model(model, time_limit=time_limit, is_mip=False)
    model.optimize()
    return extract_solution_metrics(model)


def solve_mip_model(model, time_limit):
    configure_model(model, time_limit=time_limit, is_mip=True)
    model.optimize()
    return extract_solution_metrics(model)


def greedy_schedule(instance):
    """Construct a feasible schedule for warm starts and for an integer upper bound on used slots."""
    m, R, r, edges = instance["m"], instance["R"], instance["r"], instance["edges"]
    adj = build_adjacency(m, edges)
    order = sorted(range(m), key=lambda i: (-len(adj[i]), -r[i], i))
    slot_loads = []
    slot_sets = []
    assignment = [-1] * m
    for i in order:
        placed = False
        for s, exams in enumerate(slot_sets):
            if slot_loads[s] + r[i] <= R and all((j not in adj[i]) for j in exams):
                exams.add(i)
                slot_loads[s] += r[i]
                assignment[i] = s
                placed = True
                break
        if not placed:
            slot_sets.append({i})
            slot_loads.append(r[i])
            assignment[i] = len(slot_sets) - 1
    groups = [tuple(sorted(g)) for g in slot_sets]
    return assignment, groups


def mip_slot_count(instance):
    """Return the slot count used in F1/F2 MIP models.

    If a greedy schedule uses U slots, any optimal integer solution uses at most U slots. Therefore,
    replacing n by U in the integer model is an equivalent optimization problem with fewer variables.
    LP relaxations are still solved with the original n to report the relaxation of the stated formulation.
    """
    if not USE_GREEDY_SLOT_UPPER_BOUND_FOR_MIP:
        return instance["n"]
    _, groups = greedy_schedule(instance)
    return min(instance["n"], len(groups))


def add_symmetry_breaking(model, y, n):
    for j in range(n - 1):
        model.addConstr(y[j] >= y[j + 1], name=f"sym_use_{j}")


def set_warm_start_assignment(x, y, assignment, n):
    used = {s for s in assignment if s < n}
    for j in range(n):
        y[j].Start = 1.0 if j in used else 0.0
    for i, s in enumerate(assignment):
        for j in range(n):
            x[i, j].Start = 1.0 if j == s else 0.0

## Question 1(a): Natural ILP formulation + implementation

This section implements **F1**, the natural assignment formulation.

The report gives the full mathematical formulation. In this notebook, the implementation follows the same structure:

- `x[i, j] = 1` if exam `i` is assigned to slot `j`.
- `y[j] = 1` if slot `j` is used.
- The objective minimizes the number of used slots.
- Each exam is assigned exactly once.
- Each used slot must satisfy the proctor-capacity limit.
- The linking constraint `x[i, j] <= y[j]` is included to strengthen the LP relaxation and to match the report.
- For every conflict edge `(i, k)`, the model prevents exams `i` and `k` from being assigned to the same slot.
- Symmetry-breaking constraints `y[j] >= y[j+1]` are added because slot labels are interchangeable.

For MIP solves, the code uses a greedy feasible schedule to obtain an upper bound `U` on the number of slots required. The MIP model then uses only slots `1, ..., U`. This does not change the integer optimum, because any optimal solution needs no more slots than a known feasible schedule.

In [3]:
def build_F1_natural(instance, relax=False, add_warm_start=True, active_slots=None):
    """Build the natural assignment formulation F1."""
    m, R, r, edges = instance["m"], instance["R"], instance["r"], instance["edges"]
    n = instance["n"] if active_slots is None else int(active_slots)
    vtype = GRB.CONTINUOUS if relax else GRB.BINARY
    model = gp.Model("F1_natural_LP" if relax else "F1_natural_MIP")

    x = model.addVars(m, n, lb=0, ub=1, vtype=vtype, name="x")
    y = model.addVars(n, lb=0, ub=1, vtype=vtype, name="y")

    model.setObjective(gp.quicksum(y[j] for j in range(n)), GRB.MINIMIZE)

    # Each exam is assigned exactly once.
    model.addConstrs((gp.quicksum(x[i, j] for j in range(n)) == 1 for i in range(m)), name="assign")

    # Proctor capacity in each time slot.
    model.addConstrs((gp.quicksum(r[i] * x[i, j] for i in range(m)) <= R * y[j] for j in range(n)), name="capacity")

    # Assignment-to-slot linking.
    model.addConstrs((x[i, j] <= y[j] for i in range(m) for j in range(n)), name="link")

    # Pairwise conflict constraints.
    model.addConstrs((x[i, j] + x[k, j] <= y[j] for (i, k) in edges for j in range(n)), name="conflict")

    add_symmetry_breaking(model, y, n)

    if (not relax) and add_warm_start:
        assignment, _ = greedy_schedule(instance)
        set_warm_start_assignment(x, y, assignment, n)

    model.update()
    return model

## Question 1(b): Clique-strengthened formulation + implementation

This section implements **F2**, the clique-strengthened assignment formulation.

The report explains the full clique-strengthened formulation. In this notebook, the implementation uses the following practical version:

- Start from F1.
- Keep all original pairwise conflict constraints.
- Add selected clique inequalities for larger cliques in the conflict graph.
- Use the same linking constraints `x[i, j] <= y[j]`.
- Use the same symmetry-breaking constraints `y[j] >= y[j+1]`.

A clique is a set of exams that are pairwise conflicting. Therefore, in any one slot, at most one exam from the clique can be scheduled. These clique inequalities are valid strengthening constraints for the LP relaxation.

The implementation does **not** add every maximal clique. A full maximal-clique formulation can become too large on medium and large instances. Instead, the code selects larger clique inequalities deterministically and adds them on top of all pairwise conflict constraints. This keeps the model computationally tractable while still implementing clique-based strengthening.

In [4]:
def clique_limit_for_instance(instance):
    path = instance.get("path", "")
    if "dataset2" in path.replace("\\", "/").lower():
        return MAX_CLIQUES_DATASET2
    return MAX_CLIQUES_DATASET1


def greedy_extend_clique(seed, adj, degrees):
    """Greedily extend an initial clique seed into a larger valid clique."""
    C = set(seed)
    candidates = set.intersection(*(adj[v] for v in C)) if C else set()
    while candidates:
        v = max(candidates, key=lambda u: (degrees[u], -u))
        C.add(v)
        candidates &= adj[v]
    return tuple(sorted(C))


def generate_selected_cliques(instance, max_cliques=None, min_size=MIN_CLIQUE_SIZE):
    """Generate a deterministic, bounded set of valid clique inequalities.

    This avoids enumerating every maximal clique. We seed cliques from high-degree vertices and edges,
    greedily extend them, remove duplicates, and keep the largest cliques first.
    """
    m, edges = instance["m"], instance["edges"]
    adj = build_adjacency(m, edges)
    degrees = [len(adj[i]) for i in range(m)]
    max_cliques = clique_limit_for_instance(instance) if max_cliques is None else max_cliques

    cliques = set()

    # Vertex seeds capture large local conflict groups.
    for v in sorted(range(m), key=lambda i: (-degrees[i], i)):
        C = greedy_extend_clique((v,), adj, degrees)
        if len(C) >= min_size:
            cliques.add(C)

    # Edge seeds add more variety while remaining cheap.
    sorted_edges = sorted(edges, key=lambda e: (-(degrees[e[0]] + degrees[e[1]]), e))
    for i, k in sorted_edges:
        C = greedy_extend_clique((i, k), adj, degrees)
        if len(C) >= min_size:
            cliques.add(C)
        if len(cliques) >= max_cliques * 3:
            break

    cliques = sorted(cliques, key=lambda C: (-len(C), C))[:max_cliques]
    return cliques


_CLIQUE_CACHE = {}

def get_selected_cliques(instance):
    key = (instance["path"], instance["m"], len(instance["edges"]), clique_limit_for_instance(instance))
    if key not in _CLIQUE_CACHE:
        t0 = time.time()
        cliques = generate_selected_cliques(instance)
        _CLIQUE_CACHE[key] = {"cliques": cliques, "time": time.time() - t0}
    return _CLIQUE_CACHE[key]


def build_F2_clique(instance, relax=False, add_warm_start=True, active_slots=None):
    """Build F2: natural formulation plus selected clique inequalities."""
    m, R, r, edges = instance["m"], instance["R"], instance["r"], instance["edges"]
    n = instance["n"] if active_slots is None else int(active_slots)
    clique_info = get_selected_cliques(instance)
    cliques = clique_info["cliques"]

    vtype = GRB.CONTINUOUS if relax else GRB.BINARY
    model = gp.Model("F2_clique_LP" if relax else "F2_clique_MIP")

    x = model.addVars(m, n, lb=0, ub=1, vtype=vtype, name="x")
    y = model.addVars(n, lb=0, ub=1, vtype=vtype, name="y")

    model.setObjective(gp.quicksum(y[j] for j in range(n)), GRB.MINIMIZE)

    model.addConstrs((gp.quicksum(x[i, j] for j in range(n)) == 1 for i in range(m)), name="assign")
    model.addConstrs((gp.quicksum(r[i] * x[i, j] for i in range(m)) <= R * y[j] for j in range(n)), name="capacity")
    model.addConstrs((x[i, j] <= y[j] for i in range(m) for j in range(n)), name="link")

    # Keep every original edge conflict. This guarantees that selecting only some cliques never loses validity.
    model.addConstrs((x[i, j] + x[k, j] <= y[j] for (i, k) in edges for j in range(n)), name="conflict")

    # Add selected clique strengthening constraints.
    for c_idx, C in enumerate(cliques):
        model.addConstrs((gp.quicksum(x[i, j] for i in C) <= y[j] for j in range(n)), name=f"clique_{c_idx}")

    add_symmetry_breaking(model, y, n)

    if (not relax) and add_warm_start:
        assignment, _ = greedy_schedule(instance)
        set_warm_start_assignment(x, y, assignment, n)

    model.update()
    return model, clique_info

## Question 2(a): Extended formulation + implementation

This section implements **F3**, the extended grouping formulation.

The report gives the full mathematical formulation. In this notebook, the implementation works as follows:

- First enumerate all feasible nonempty exam groupings.
- A grouping is feasible if:
  - its total proctor requirement is at most `R`;
  - it is a stable set in the conflict graph, meaning no two exams in the grouping have a conflict edge.
- Create one binary variable `z[g]` for each feasible grouping `g`.
- `z[g] = 1` means that grouping `g` is selected as one used time slot.
- The objective minimizes the number of selected groupings.
- Each exam must appear in exactly one selected grouping, so this is the set-partitioning version of the extended formulation.

This formulation does not need separate slot-capacity or pairwise conflict constraints, because each grouping is already feasible by construction.

F3 is run only on Dataset 1. It is not run on Dataset 2 because the number of feasible groupings can grow exponentially with the number of exams, making enumeration impractical for the larger instances.

In [5]:
def enumerate_feasible_groupings(instance):
    """Enumerate all feasible nonempty groups: stable sets with total proctor load <= R.

    The recursion builds groups in increasing exam index order. A candidate can be added only if it is not
    adjacent to any already selected exam and if the capacity remains feasible.
    """
    m, R, r, edges = instance["m"], instance["R"], instance["r"], instance["edges"]
    adj_masks = build_adj_masks(m, edges)
    groups = []

    def backtrack(start, selected, selected_mask, load):
        for i in range(start, m):
            if load + r[i] > R:
                continue
            if selected_mask & adj_masks[i]:
                continue
            new_selected = selected + [i]
            new_mask = selected_mask | (1 << i)
            new_load = load + r[i]
            groups.append(tuple(new_selected))
            backtrack(i + 1, new_selected, new_mask, new_load)

    backtrack(0, [], 0, 0)
    groups = sorted(set(groups), key=lambda g: (len(g), g))
    return groups


_GROUP_CACHE = {}

def get_feasible_groupings(instance):
    key = (instance["path"], instance["m"], len(instance["edges"]), instance["R"])
    if key not in _GROUP_CACHE:
        t0 = time.time()
        groups = enumerate_feasible_groupings(instance)
        _GROUP_CACHE[key] = {"groups": groups, "time": time.time() - t0}
    return _GROUP_CACHE[key]


def build_F3_extended(instance, relax=False, add_warm_start=True):
    """Build the extended grouping formulation F3."""
    m = instance["m"]
    group_info = get_feasible_groupings(instance)
    groups = group_info["groups"]
    group_to_idx = {g: idx for idx, g in enumerate(groups)}

    vtype = GRB.CONTINUOUS if relax else GRB.BINARY
    model = gp.Model("F3_extended_LP" if relax else "F3_extended_MIP")

    z = model.addVars(len(groups), lb=0, ub=1, vtype=vtype, name="z")
    model.setObjective(gp.quicksum(z[g] for g in range(len(groups))), GRB.MINIMIZE)

    groups_by_exam = [[] for _ in range(m)]
    for g_idx, group in enumerate(groups):
        for i in group:
            groups_by_exam[i].append(g_idx)

    model.addConstrs((gp.quicksum(z[g_idx] for g_idx in groups_by_exam[i]) == 1 for i in range(m)), name="cover_exactly_once")

    if (not relax) and add_warm_start:
        _, greedy_groups = greedy_schedule(instance)
        for g_idx in range(len(groups)):
            z[g_idx].Start = 0.0
        for group in greedy_groups:
            idx = group_to_idx.get(tuple(group))
            if idx is not None:
                z[idx].Start = 1.0

    model.update()
    return model, group_info

## Question 2(b): Set covering vs. set partitioning discussion

This section summarizes the modeling choice used for the extended formulation.

There are two natural ways to write the grouping-based formulation:

### Set partitioning

The set-partitioning version requires each exam to appear in exactly one selected grouping.

This matches the scheduling problem directly, because each exam must be assigned to exactly one time slot. This is the version implemented in this notebook for F3.

### Set covering

The set-covering version requires each exam to appear in at least one selected grouping.

This version is less exact as a direct schedule description because the same exam may be covered by more than one selected grouping. However, for integer solutions, the optimal objective value can be the same as the set-partitioning version when all feasible groupings and their feasible subsets are included. If an exam is covered more than once, duplicate membership can be removed from selected groupings without increasing the number of selected slots. If a grouping becomes empty after removal, it can be deleted.

### Choice used in this notebook

This notebook uses **set partitioning** for F3 because it directly encodes the requirement that every exam is scheduled exactly once. The set-covering version is useful as a related modeling alternative, especially in column-generation settings, but the set-partitioning version is the cleanest implementation for this project.

# Experiment utilities for Question 3

The following functions solve one formulation on one instance. For each formulation-instance pair, the notebook records:

- dataset name;
- instance number and size;
- formulation name;
- LP relaxation value;
- MIP objective value;
- best bound and MIP gap;
- branch-and-bound node count;
- Gurobi wall-clock solve time;
- whether the run proved optimality within the prescribed time limit.

Important: a `TimeLimit` status is a valid reported outcome, not a missing result. It means Gurobi was run under the required time limit but did not prove optimality within that limit.

In [6]:
def solve_formulation_on_instance(dataset_name, instance_number, instance, formulation, time_limit):
    """Solve LP relaxation and MIP for one formulation on one instance."""
    row = {
        "dataset": dataset_name,
        "instance": f"{instance_number:02d}",
        "m": instance["m"],
        "n": instance["n"],
        "R": instance["R"],
        "num_edges": len(instance["edges"]),
        "formulation": formulation,
        "time_limit": time_limit,
    }

    # LP relaxation: report the relaxation of the stated formulation using the original n slots.
    t_build_lp = time.time()
    if formulation == "F1":
        lp_model = build_F1_natural(instance, relax=True, add_warm_start=False)
        aux_info = {}
    elif formulation == "F2":
        lp_model, clique_info = build_F2_clique(instance, relax=True, add_warm_start=False)
        aux_info = {"num_cliques": len(clique_info["cliques"]), "clique_generation_time": clique_info["time"]}
    elif formulation == "F3":
        lp_model, group_info = build_F3_extended(instance, relax=True, add_warm_start=False)
        aux_info = {"num_groupings": len(group_info["groups"]), "group_enum_time": group_info["time"]}
    else:
        raise ValueError(f"Unknown formulation: {formulation}")
    row["lp_model_build_time"] = time.time() - t_build_lp
    row.update(aux_info)

    lp_metrics = solve_lp_model(lp_model, time_limit=LP_TIME_LIMIT)
    row["lp_status"] = lp_metrics["status"]
    row["lp_relaxation_value"] = lp_metrics["objective"]
    row["lp_solve_time"] = lp_metrics["solve_time"]

    # MIP: for F1/F2 use the greedy slot upper bound to reduce equivalent slot variables.
    t_build_mip = time.time()
    if formulation == "F1":
        active_slots = mip_slot_count(instance)
        mip_model = build_F1_natural(instance, relax=False, add_warm_start=True, active_slots=active_slots)
        row["active_slots_mip"] = active_slots
    elif formulation == "F2":
        active_slots = mip_slot_count(instance)
        mip_model, _ = build_F2_clique(instance, relax=False, add_warm_start=True, active_slots=active_slots)
        row["active_slots_mip"] = active_slots
    elif formulation == "F3":
        mip_model, _ = build_F3_extended(instance, relax=False, add_warm_start=True)
        row["active_slots_mip"] = None
    row["mip_model_build_time"] = time.time() - t_build_mip

    mip_metrics = solve_mip_model(mip_model, time_limit=time_limit)
    row["mip_status"] = mip_metrics["status"]
    row["sol_count"] = mip_metrics["sol_count"]
    row["mip_objective"] = mip_metrics["objective"]
    row["best_bound"] = mip_metrics["best_bound"]
    row["mip_gap"] = mip_metrics["mip_gap"]
    row["node_count"] = mip_metrics["node_count"]
    row["solve_time"] = mip_metrics["solve_time"]
    row["solved_to_optimality"] = mip_metrics["optimality"]

    return row


def read_checkpoint(path):
    if RESUME_FROM_CHECKPOINTS and Path(path).exists():
        return pd.read_csv(path)
    return pd.DataFrame()


def run_experiment(dataset_name, dataset_dir, instance_numbers, formulations, time_limit, checkpoint_path):
    checkpoint_path = Path(checkpoint_path)
    existing = read_checkpoint(checkpoint_path)
    completed = set()
    rows = []
    if not existing.empty:
        for _, r in existing.iterrows():
            completed.add((str(r["dataset"]), str(r["instance"]).zfill(2), str(r["formulation"])))
        rows = existing.to_dict("records")
        print(f"Loaded {len(existing)} checkpoint rows from {checkpoint_path}")

    print("=" * 80)
    print(f"RUNNING {dataset_name.upper()} | time limit = {time_limit}s per MIP solve")
    print("=" * 80)

    for k in instance_numbers:
        path = instance_path(dataset_dir, k)
        if not path.exists():
            raise FileNotFoundError(f"Missing required instance file: {path}")
        instance = load_instance(path)
        print(f"\nInstance {k:02d}: m={instance['m']}, |E|={len(instance['edges'])}")

        for formulation in formulations:
            key = (dataset_name, f"{k:02d}", formulation)
            if key in completed:
                print(f"  {formulation}: already in checkpoint; skipping recomputation.")
                continue
            print(f"  Solving {formulation} ...", flush=True)
            row = solve_formulation_on_instance(dataset_name, k, instance, formulation, time_limit)
            rows.append(row)
            pd.DataFrame(rows).to_csv(checkpoint_path, index=False)

            lpv = row.get("lp_relaxation_value")
            obj = row.get("mip_objective")
            nodes = row.get("node_count")
            st = row.get("mip_status")
            t = row.get("solve_time")
            print(f"    LP={lpv}  MIP={obj}  nodes={nodes}  t={t:.1f}s  status={st}")

    final_df = pd.DataFrame(rows)
    final_df.to_csv(checkpoint_path, index=False)
    return final_df


def display_project_table(df):
    cols = [
        "dataset", "instance", "m", "num_edges", "formulation",
        "lp_status", "lp_relaxation_value", "mip_objective", "best_bound", "mip_gap",
        "node_count", "solve_time", "mip_status", "solved_to_optimality",
        "active_slots_mip", "num_cliques", "num_groupings"
    ]
    present = [c for c in cols if c in df.columns]
    out = df[present].copy()
    for c in ["lp_relaxation_value", "mip_objective", "best_bound", "mip_gap", "node_count", "solve_time"]:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce")
    return out.sort_values(["dataset", "instance", "formulation"]).reset_index(drop=True)

# Question 3(a): Dataset 1 computational results + comparison discussion

## What this question asks

Dataset 1 contains 10 instances of increasing size. The project requires solving all three formulations on every Dataset 1 instance:

- F1: natural formulation from Question 1(a);
- F2: clique-strengthened formulation from Question 1(b);
- F3: extended formulation from Question 2(a).

For each formulation and instance, we report the LP relaxation value, MIP objective, MIP gap, branch-and-bound node count, Gurobi wall-clock solve time, and whether Gurobi proved optimality within the 120-second time limit.

A `TimeLimit` status is acceptable and should be interpreted as a computational result. It means the formulation-instance pair was solved under the required time limit, but optimality was not proven before the limit.

In [7]:
if RUN_DATASET1:
    df1 = run_experiment(
        dataset_name="dataset1",
        dataset_dir=DATASET1_DIR,
        instance_numbers=DATASET1_INSTANCES,
        formulations=["F1", "F2", "F3"],
        time_limit=DATASET1_TIME_LIMIT,
        checkpoint_path=D1_CHECKPOINT,
    )
else:
    df1 = read_checkpoint(D1_CHECKPOINT)

results_dataset1 = display_project_table(df1)
results_dataset1

RUNNING DATASET1 | time limit = 120s per MIP solve

Instance 01: m=35, |E|=292
  Solving F1 ...
Set parameter Username
Set parameter LicenseID to value 2723230
Academic license - for non-commercial use only - expires 2026-10-17
    LP=6.320000000000005  MIP=8.0  nodes=1.0  t=1.4s  status=Optimal
  Solving F2 ...
    LP=6.319999999999995  MIP=8.0  nodes=1.0  t=2.5s  status=Optimal
  Solving F3 ...
    LP=7.1833333333333345  MIP=8.0  nodes=1.0  t=0.8s  status=Optimal

Instance 02: m=38, |E|=345
  Solving F1 ...
    LP=7.320000000000013  MIP=9.0  nodes=427.0  t=7.2s  status=Optimal
  Solving F2 ...
    LP=7.319999999999993  MIP=9.0  nodes=517.0  t=10.4s  status=Optimal
  Solving F3 ...
    LP=7.920634920634921  MIP=9.0  nodes=1.0  t=1.2s  status=Optimal

Instance 03: m=40, |E|=369
  Solving F1 ...
    LP=7.880000000000003  MIP=9.0  nodes=137.0  t=6.6s  status=Optimal
  Solving F2 ...
    LP=7.879999999999988  MIP=9.0  nodes=231.0  t=4.6s  status=Optimal
  Solving F3 ...
    LP=8.118712273

,dataset,instance,m,num_edges,formulation,lp_status,lp_relaxation_value,mip_objective,best_bound,mip_gap,node_count,solve_time,mip_status,solved_to_optimality,active_slots_mip,num_cliques,num_groupings
0,dataset1,01,35,292,F1,Optimal,6.320000,8.0,8.0,0.000000,1.0,1.370,Optimal,True,9.0,NaN,NaN
1,dataset1,01,35,292,F2,Optimal,6.320000,8.0,8.0,0.000000,1.0,2.498,Optimal,True,9.0,143.0,NaN
2,dataset1,01,35,292,F3,Optimal,7.183333,8.0,8.0,0.000000,1.0,0.822,Optimal,True,NaN,NaN,2573.0
3,dataset1,02,38,345,F1,Optimal,7.320000,9.0,9.0,0.000000,427.0,7.174,Optimal,True,10.0,NaN,NaN
4,dataset1,02,38,345,F2,Optimal,7.320000,9.0,9.0,0.000000,517.0,10.423,Optimal,True,10.0,169.0,NaN
5,dataset1,02,38,345,F3,Optimal,7.920635,9.0,9.0,0.000000,1.0,1.195,Optimal,True,NaN,NaN,2809.0
6,dataset1,03,40,369,F1,Optimal,7.880000,9.0,9.0,0.000000,137.0,6.557,Optimal,True,11.0,NaN,NaN
7,dataset1,03,40,369,F2,Optimal,7.880000,9.0,9.0,0.000000,231.0,4.617,Optimal,True,11.0,187.0,NaN
8,dataset1,03,40,369,F3,Optimal,8.118712,9.0,9.0,0.000000,1.0,2.086,Optimal,True,NaN,NaN,4209.0
9,dataset1,04,60,886,F1,Optimal,12.840000,13.0,13.0,0.000000,1093.0,31.241,Optimal,True,15.0,NaN,NaN


In [8]:
# Dataset 1 compact comparison table: one row per instance, with LP values, objectives, nodes, and times by formulation.
if not df1.empty:
    d1_pivot = df1.pivot_table(
        index=["dataset", "instance", "m", "num_edges"],
        columns="formulation",
        values=["lp_relaxation_value", "mip_objective", "node_count", "solve_time"],
        aggfunc="first"
    )
    d1_pivot

In [9]:
def generate_dataset1_discussion(df):
    if df.empty:
        return "Dataset 1 has not been run yet."
    lines = []
    lines.append("## Dataset 1 discussion generated from the result table\n")

    pivot_lp = df.pivot_table(index="instance", columns="formulation", values="lp_relaxation_value", aggfunc="first")
    if {"F1", "F2"}.issubset(pivot_lp.columns):
        diff_f2 = (pivot_lp["F2"] - pivot_lp["F1"]).dropna()
        improved = int((diff_f2 > 1e-6).sum())
        equal = int((diff_f2.abs() <= 1e-6).sum())
        lines.append(f"Across Dataset 1, F2 has a strictly better LP bound than F1 in {improved} instances and the same LP bound in {equal} instances. This measures the marginal effect of clique strengthening over the natural edge-based conflict model.")
    if {"F1", "F3"}.issubset(pivot_lp.columns):
        diff_f3 = (pivot_lp["F3"] - pivot_lp["F1"]).dropna()
        improved3 = int((diff_f3 > 1e-6).sum())
        equal3 = int((diff_f3.abs() <= 1e-6).sum())
        lines.append(f"F3 has a strictly better LP bound than F1 in {improved3} instances and the same LP bound in {equal3} instances. When it improves the LP bound, the improvement comes from representing one whole feasible time-slot pattern at a time rather than using only assignment and conflict inequalities.")

    opt = df.groupby("formulation")["solved_to_optimality"].sum().to_dict()
    total = df.groupby("formulation")["instance"].count().to_dict()
    for f in sorted(total):
        lines.append(f"{f} solves {int(opt.get(f, 0))} out of {int(total[f])} Dataset 1 instances to proven optimality within the 120-second time limit.")

    avg_nodes = df.groupby("formulation")["node_count"].mean(numeric_only=True).to_dict()
    avg_time = df.groupby("formulation")["solve_time"].mean(numeric_only=True).to_dict()
    for f in sorted(avg_time):
        lines.append(f"Average B&B nodes for {f}: {avg_nodes.get(f, float('nan')):.1f}; average MIP solve time: {avg_time.get(f, float('nan')):.1f} seconds.")

    lines.append("Equal or very similar LP relaxation values do not necessarily imply similar branch-and-bound performance. Branch-and-bound is also affected by model size, variable structure, branching decisions, cut generation, primal heuristics, and symmetry. Therefore, two formulations can have the same root LP objective but still explore different numbers of nodes or require different solve times.")
    lines.append("As instance size grows, computational effort generally increases because the number of assignment variables, conflict/clique constraints, and possible branching decisions all grow. F3 can be strong on Dataset 1 because the enumerated grouping model often gives a compact branch-and-bound search once the feasible groupings have been generated. However, this advantage depends on the number of feasible groupings remaining manageable.")
    return "\n\n".join(lines)

print(generate_dataset1_discussion(df1))

## Dataset 1 discussion generated from the result table


Across Dataset 1, F2 has a strictly better LP bound than F1 in 0 instances and the same LP bound in 10 instances. This measures the marginal effect of clique strengthening over the natural edge-based conflict model.

F3 has a strictly better LP bound than F1 in 3 instances and the same LP bound in 7 instances. When it improves the LP bound, the improvement comes from representing one whole feasible time-slot pattern at a time rather than using only assignment and conflict inequalities.

F1 solves 9 out of 10 Dataset 1 instances to proven optimality within the 120-second time limit.

F2 solves 7 out of 10 Dataset 1 instances to proven optimality within the 120-second time limit.

F3 solves 10 out of 10 Dataset 1 instances to proven optimality within the 120-second time limit.

Average B&B nodes for F1: 687.2; average MIP solve time: 27.1 seconds.

Average B&B nodes for F2: 859.8; average MIP solve time: 53.1 seconds.

Average B&B

# Question 3(b): Dataset 2 computational results + comparison discussion

## What this question asks

Dataset 2 contains larger instances. The project requires solving only:

- F1: natural formulation;
- F2: clique-strengthened formulation.

F3 is not run on Dataset 2 because it requires enumerating all feasible exam groupings. The number of such groupings can grow combinatorially with the number of exams, so enumeration is not practical for Dataset 2.

The same result fields are reported as in Dataset 1. The comparison focuses on how F1 and F2 behave as instance size grows under the 600-second time limit.

In [10]:
if RUN_DATASET2:
    df2 = run_experiment(
        dataset_name="dataset2",
        dataset_dir=DATASET2_DIR,
        instance_numbers=DATASET2_INSTANCES,
        formulations=["F1", "F2"],
        time_limit=DATASET2_TIME_LIMIT,
        checkpoint_path=D2_CHECKPOINT,
    )
else:
    df2 = read_checkpoint(D2_CHECKPOINT)

results_dataset2 = display_project_table(df2)
results_dataset2

RUNNING DATASET2 | time limit = 600s per MIP solve

Instance 01: m=80, |E|=1584
  Solving F1 ...
    LP=16.319999999999983  MIP=17.0  nodes=1.0  t=28.1s  status=Optimal
  Solving F2 ...
    LP=16.32000000000001  MIP=17.0  nodes=1.0  t=47.5s  status=Optimal

Instance 02: m=82, |E|=1682
  Solving F1 ...
    LP=16.08  MIP=17.0  nodes=1.0  t=30.6s  status=Optimal
  Solving F2 ...
    LP=16.080000000000027  MIP=17.0  nodes=1.0  t=26.2s  status=Optimal

Instance 03: m=83, |E|=1726
  Solving F1 ...
    LP=16.280000000000015  MIP=17.0  nodes=1.0  t=41.2s  status=Optimal
  Solving F2 ...
    LP=16.27999999999998  MIP=17.0  nodes=1.0  t=32.3s  status=Optimal

Instance 04: m=87, |E|=1843
  Solving F1 ...
    LP=16.759999999999998  MIP=18.0  nodes=18712.0  t=600.0s  status=TimeLimit
  Solving F2 ...
    LP=16.760000000000048  MIP=17.0  nodes=1.0  t=193.1s  status=Optimal

Instance 05: m=87, |E|=1867
  Solving F1 ...
    LP=16.599999999999994  MIP=17.0  nodes=6812.0  t=343.2s  status=Optimal
  Solv

,dataset,instance,m,num_edges,formulation,lp_status,lp_relaxation_value,mip_objective,best_bound,mip_gap,node_count,solve_time,mip_status,solved_to_optimality,active_slots_mip,num_cliques
0,dataset2,01,80,1584,F1,Optimal,16.32,17.0,17.0,0.000000,1.0,28.059,Optimal,True,19,NaN
1,dataset2,01,80,1584,F2,Optimal,16.32,17.0,17.0,0.000000,1.0,47.503,Optimal,True,19,500.0
2,dataset2,02,82,1682,F1,Optimal,16.08,17.0,17.0,0.000000,1.0,30.565,Optimal,True,20,NaN
3,dataset2,02,82,1682,F2,Optimal,16.08,17.0,17.0,0.000000,1.0,26.156,Optimal,True,20,500.0
4,dataset2,03,83,1726,F1,Optimal,16.28,17.0,17.0,0.000000,1.0,41.240,Optimal,True,20,NaN
5,dataset2,03,83,1726,F2,Optimal,16.28,17.0,17.0,0.000000,1.0,32.255,Optimal,True,20,500.0
6,dataset2,04,87,1843,F1,Optimal,16.76,18.0,17.0,0.055556,18712.0,600.046,TimeLimit,False,19,NaN
7,dataset2,04,87,1843,F2,Optimal,16.76,17.0,17.0,0.000000,1.0,193.096,Optimal,True,19,500.0
8,dataset2,05,87,1867,F1,Optimal,16.60,17.0,17.0,0.000000,6812.0,343.219,Optimal,True,21,NaN
9,dataset2,05,87,1867,F2,Optimal,16.60,17.0,17.0,0.000000,1.0,133.911,Optimal,True,21,500.0


In [11]:
# Dataset 2 compact comparison table: one row per instance, with LP values, objectives, nodes, and times by formulation.
if not df2.empty:
    d2_pivot = df2.pivot_table(
        index=["dataset", "instance", "m", "num_edges"],
        columns="formulation",
        values=["lp_relaxation_value", "mip_objective", "node_count", "solve_time"],
        aggfunc="first"
    )
    d2_pivot

In [12]:
def generate_dataset2_discussion(df):
    if df.empty:
        return "Dataset 2 has not been run yet."
    lines = []
    lines.append("## Dataset 2 discussion generated from the result table\n")

    pivot_lp = df.pivot_table(index="instance", columns="formulation", values="lp_relaxation_value", aggfunc="first")
    if {"F1", "F2"}.issubset(pivot_lp.columns):
        diff = (pivot_lp["F2"] - pivot_lp["F1"]).dropna()
        improved = int((diff > 1e-6).sum())
        equal = int((diff.abs() <= 1e-6).sum())
        lines.append(f"In Dataset 2, F2 has a strictly better LP bound than F1 in {improved} instances and the same LP bound in {equal} instances. This indicates whether maximal-clique inequalities materially strengthen the edge-based natural formulation on the larger instances.")

    opt = df.groupby("formulation")["solved_to_optimality"].sum().to_dict()
    total = df.groupby("formulation")["instance"].count().to_dict()
    for f in sorted(total):
        lines.append(f"{f} solves {int(opt.get(f, 0))} out of {int(total[f])} Dataset 2 instances to proven optimality within the 600-second time limit.")

    avg_nodes = df.groupby("formulation")["node_count"].mean(numeric_only=True).to_dict()
    avg_time = df.groupby("formulation")["solve_time"].mean(numeric_only=True).to_dict()
    for f in sorted(avg_time):
        lines.append(f"Average B&B nodes for {f}: {avg_nodes.get(f, float('nan')):.1f}; average MIP solve time: {avg_time.get(f, float('nan')):.1f} seconds.")

    lines.append("The computational limit of F1 and F2 becomes more visible as instance size grows. F1 has a relatively simple structure but can have a weak LP relaxation. F2 can strengthen the LP relaxation through clique inequalities, but it also introduces more constraints and can be more expensive per LP solve and per branch-and-bound node.")
    lines.append("F3 is not applied to Dataset 2 because it requires enumerating all feasible stable sets under the proctor capacity. Even with the capacity bound, the number of feasible groupings can grow combinatorially with the number of exams. For larger instances, this enumeration can dominate the computation before the optimization model is even solved.")
    lines.append("The relationship between LP bound quality and branch-and-bound performance is positive but not mechanical. A stronger LP bound can reduce the search tree, but a larger or denser formulation can increase root LP time, node processing time, and memory burden. The final comparison must therefore consider LP value, node count, and solve time together.")
    return "\n\n".join(lines)

print(generate_dataset2_discussion(df2))

## Dataset 2 discussion generated from the result table


In Dataset 2, F2 has a strictly better LP bound than F1 in 0 instances and the same LP bound in 9 instances. This indicates whether maximal-clique inequalities materially strengthen the edge-based natural formulation on the larger instances.

F1 solves 7 out of 9 Dataset 2 instances to proven optimality within the 600-second time limit.

F2 solves 6 out of 9 Dataset 2 instances to proven optimality within the 600-second time limit.

Average B&B nodes for F1: 6387.0; average MIP solve time: 255.4 seconds.

Average B&B nodes for F2: 5954.3; average MIP solve time: 294.4 seconds.

The computational limit of F1 and F2 becomes more visible as instance size grows. F1 has a relatively simple structure but can have a weak LP relaxation. F2 can strengthen the LP relaxation through clique inequalities, but it also introduces more constraints and can be more expensive per LP solve and per branch-and-bound node.

F3 is not applied to Datase

# Final combined result table

The table below combines Dataset 1 and Dataset 2 results in the format required by the project. Dataset 1 includes F1, F2, and F3. Dataset 2 includes only F1 and F2.

In [13]:
all_results = pd.concat([df1, df2], ignore_index=True) if ('df1' in globals() and 'df2' in globals()) else pd.DataFrame()
final_results_table = display_project_table(all_results) if not all_results.empty else pd.DataFrame()
final_results_table.to_csv("final_project_results.csv", index=False)
final_results_table

,dataset,instance,m,num_edges,formulation,lp_status,lp_relaxation_value,mip_objective,best_bound,mip_gap,node_count,solve_time,mip_status,solved_to_optimality,active_slots_mip,num_cliques,num_groupings
0,dataset1,01,35,292,F1,Optimal,6.320000,8.0,8.0,0.000000,1.0,1.370,Optimal,True,9.0,NaN,NaN
1,dataset1,01,35,292,F2,Optimal,6.320000,8.0,8.0,0.000000,1.0,2.498,Optimal,True,9.0,143.0,NaN
2,dataset1,01,35,292,F3,Optimal,7.183333,8.0,8.0,0.000000,1.0,0.822,Optimal,True,NaN,NaN,2573.0
3,dataset1,02,38,345,F1,Optimal,7.320000,9.0,9.0,0.000000,427.0,7.174,Optimal,True,10.0,NaN,NaN
4,dataset1,02,38,345,F2,Optimal,7.320000,9.0,9.0,0.000000,517.0,10.423,Optimal,True,10.0,169.0,NaN
5,dataset1,02,38,345,F3,Optimal,7.920635,9.0,9.0,0.000000,1.0,1.195,Optimal,True,NaN,NaN,2809.0
6,dataset1,03,40,369,F1,Optimal,7.880000,9.0,9.0,0.000000,137.0,6.557,Optimal,True,11.0,NaN,NaN
7,dataset1,03,40,369,F2,Optimal,7.880000,9.0,9.0,0.000000,231.0,4.617,Optimal,True,11.0,187.0,NaN
8,dataset1,03,40,369,F3,Optimal,8.118712,9.0,9.0,0.000000,1.0,2.086,Optimal,True,NaN,NaN,4209.0
9,dataset1,04,60,886,F1,Optimal,12.840000,13.0,13.0,0.000000,1093.0,31.241,Optimal,True,15.0,NaN,NaN
